In [12]:
from pathlib import Path
import pickle
import cv2
import numpy as np

# =========================
# AJUSTES
# =========================
ROI_W = 0.78
ROI_H = 0.78

MIN_AREA = 1800
BLOCK_SIZE = 31   # impar: 21, 31, 41...
C = 7             # 5, 7, 9...
KERNEL_SIZE = 5   # 3 o 5
MATCH_THRESHOLD = 0.18


# =========================
# ROI central
# =========================
def get_center_roi(frame, roi_w=ROI_W, roi_h=ROI_H):
    h, w = frame.shape[:2]

    rw = int(w * roi_w)
    rh = int(h * roi_h)

    x1 = (w - rw) // 2
    y1 = (h - rh) // 2
    x2 = x1 + rw
    y2 = y1 + rh

    roi = frame[y1:y2, x1:x2]
    return (x1, y1, x2, y2), roi


# =========================
# Preprocesamiento
# =========================
def preprocess_roi(roi, block_size=BLOCK_SIZE, c=C, kernel_size=KERNEL_SIZE):
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 5)

    mask = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        block_size,
        c
    )

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (kernel_size, kernel_size)
    )

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    return gray, mask


# =========================
# Contornos válidos
# =========================
def get_valid_contours(mask, min_area=MIN_AREA):
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    h, w = mask.shape[:2]
    valid = []

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue

        x, y, cw, ch = cv2.boundingRect(cnt)

        # ignorar objetos pegados al borde de la ROI
        if x <= 2 or y <= 2 or (x + cw) >= (w - 2) or (y + ch) >= (h - 2):
            continue

        valid.append(cnt)

    valid.sort(key=cv2.contourArea, reverse=True)
    return valid


# =========================
# Clasificación con PKL
# =========================
def classify_shape_from_templates(cnt, templates, threshold=MATCH_THRESHOLD):
    best_label = "unknown"
    best_score = float("inf")

    for label, tmpl in templates.items():
        score = cv2.matchShapes(cnt, tmpl, cv2.CONTOURS_MATCH_I1, 0.0)

        if score < best_score:
            best_score = score
            best_label = label

    if best_score > threshold:
        return "unknown", best_score

    return best_label, best_score

In [13]:
import pickle

with open("shape_templates.pkl", "rb") as f:
    templates = pickle.load(f)

print(templates.keys())

dict_keys(['caplet', 'pill', 'circle'])


In [14]:
for name, cnt in templates.items():
    print(name, type(cnt), cnt.shape)

caplet <class 'numpy.ndarray'> (222, 1, 2)
pill <class 'numpy.ndarray'> (144, 1, 2)
circle <class 'numpy.ndarray'> (110, 1, 2)


In [15]:
from pathlib import Path
import pickle

base = Path("pills")
pkl_path = "shape_templates.pkl"

with open(pkl_path, "rb") as f:
    templates = pickle.load(f)

print(templates.keys())

dict_keys(['caplet', 'pill', 'circle'])


In [16]:
list(Path("pills").glob("*"))

[WindowsPath('pills/.DS_Store'),
 WindowsPath('pills/317220267.jpg'),
 WindowsPath('pills/60429-203_M_LH3.jpg'),
 WindowsPath('pills/634810623.jpg'),
 WindowsPath('pills/blackNyellow_capsule.jpg'),
 WindowsPath('pills/bloe_oval.jpg'),
 WindowsPath('pills/brown_oval.jpg'),
 WindowsPath('pills/cream_capsule.jpg'),
 WindowsPath('pills/green_capsule.jpg'),
 WindowsPath('pills/green_oval.jpg'),
 WindowsPath('pills/orangeandblue.jpg')]

In [17]:
image_paths = []
for ext in ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.webp"):
    image_paths.extend(base.glob(ext))

image_paths = sorted(image_paths)

print(f"imagenes encontradas: {len(image_paths)}")
for p in image_paths[:10]:
    print(p.name)

imagenes encontradas: 10
317220267.jpg
60429-203_M_LH3.jpg
634810623.jpg
blackNyellow_capsule.jpg
bloe_oval.jpg
brown_oval.jpg
cream_capsule.jpg
green_capsule.jpg
green_oval.jpg
orangeandblue.jpg


In [18]:
import cv2
def classify_shape_from_templates(cnt, templates, threshold=0.18):
    best_label = "unknown"
    best_score = float("inf")

    for label, tmpl in templates.items():
        score = cv2.matchShapes(cnt, tmpl, cv2.CONTOURS_MATCH_I1, 0.0)
        if score < best_score:
            best_score = score
            best_label = label

    if best_score > threshold:
        return "unknown", best_score

    return best_label, best_score


results = []

for img_path in image_paths:
    img = cv2.imread(str(img_path))
    if img is None:
        results.append((img_path.name, "read_error", None))
        continue

    (_, _, _, _), roi = get_center_roi(img)
    gray, mask = preprocess_roi(roi)
    contours = get_valid_contours(mask)

    if not contours:
        results.append((img_path.name, "no_contour", None))
        continue

    biggest = contours[0]
    label, score = classify_shape_from_templates(biggest, templates)

    results.append((img_path.name, label, score))

for name, label, score in results:
    if score is None:
        print(f"{name:30s} -> {label}")
    else:
        print(f"{name:30s} -> {label:10s}  score={score:.4f}")

317220267.jpg                  -> no_contour
60429-203_M_LH3.jpg            -> no_contour
634810623.jpg                  -> no_contour
blackNyellow_capsule.jpg       -> unknown     score=0.6542
bloe_oval.jpg                  -> unknown     score=0.5696
brown_oval.jpg                 -> no_contour
cream_capsule.jpg              -> unknown     score=0.7639
green_capsule.jpg              -> unknown     score=3.3257
green_oval.jpg                 -> no_contour
orangeandblue.jpg              -> circle      score=0.0139
